## Project: Spam SMS Detection using Naive Bayes

### Introduction to Naive Bayes

Naive Bayes is a family of probabilistic classifiers based on Bayes' theorem with a 'naive' independence assumption between the features. In the context of spam detection, this means we assume that the presence of one word in an SMS is independent of the presence of any other word, given that the SMS is either spam or not spam (ham).

Despite this strong, often unrealistic, independence assumption, Naive Bayes classifiers perform remarkably well in many real-world scenarios, especially in text classification. This is largely due to their simplicity, speed, and ability to handle high-dimensional data.

**Bayes' Theorem:**

$$P(Class|Features) = \frac{P(Features|Class) * P(Class)}{P(Features)}$$

Where:
- $P(Class|Features)$ is the posterior probability: the probability of an SMS belonging to a certain class (e.g., 'spam') given the observed features (words in the SMS).
- $P(Features|Class)$ is the likelihood: the probability of observing these features given that the SMS belongs to that class.
- $P(Class)$ is the prior probability: the overall probability of that class (e.g., the proportion of spam SMS in the dataset).
- $P(Features)$ is the evidence: the overall probability of observing these features.

For spam detection, we'll calculate the probability of an SMS being spam given its words, and the probability of it being ham given its words, then classify it based on which probability is higher.

## **Loading Dataset**

In [103]:
import pandas as pd

data = pd.read_csv('SMS.csv', encoding='latin-1') #the default encoding is utf-8 but our SMS dataset likely have special characters that utf-8 encoding cannot deal with, which might cause UnicodeDecodeError when reading this file

print('Original Columns for Dataset:', data.columns)

print()

print(data.head())


Original Columns for Dataset: Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  


## **Cleaning the Dataset (Redundant columns)**

In [104]:
data = data.drop(['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], axis=1) #delete the empry useless columns

data = data.rename(columns={'v1' : 'label', 'v2' : 'message'})

print('New Columns for Dataset:', data.columns)


New Columns for Dataset: Index(['label', 'message'], dtype='object')


In [105]:
print(data['label'].value_counts()) #find out num of spam and ham SMS in dataset

label
ham     4825
spam     747
Name: count, dtype: int64


In [106]:
#Clean the messy SMS Contents

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def clean_text(text):
    text = text.lower() #make all lowercase so that capitalisation doesn't matter
    #text = re.sub(r'http\S+', 'URL', text)      #replace URLs with placeholder "URL" (make it less messy by labelling all the weird gibberish urls into just URL)
    #text = re.sub(r'\S+@\S+', 'EMAIL', text)    #replace emails with placeholder "EMAIL" (make it less messy by labelling all the weird gibberish emails them into just EMAIL)
    text = re.sub(r'[^a-z\s]', '', text) #replacing chars that are not english alphabets to empty -> delete them
    words = text.split()
    nonstopwords = []
    for i in range(len(words)):
        if words[i] not in ENGLISH_STOP_WORDS:
            nonstopwords.append(words[i])
    words = nonstopwords
    return " ".join(words)



data['clean_message'] = data['message'].apply(clean_text)

print(data[['message', 'clean_message']].head())


                                             message  \
0  Go until jurong point, crazy.. Available only ...   
1                      Ok lar... Joking wif u oni...   
2  Free entry in 2 a wkly comp to win FA Cup fina...   
3  U dun say so early hor... U c already then say...   
4  Nah I don't think he goes to usf, he lives aro...   

                                       clean_message  
0  jurong point crazy available bugis n great wor...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  
3                        u dun say early hor u c say  
4                      nah dont think goes usf lives  


## **Split into Training and Testing Sets**

In [107]:
from sklearn.model_selection import train_test_split

X = data['clean_message']
Y = data['label']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y) #stratify=Y ensures that both train and test datasets keep a similar spam-to-ham ratio


## **Build Model Pipeline + Train Model**

In [108]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

model = Pipeline([
    ('vectorizer', CountVectorizer(ngram_range=(1,2))),
    ('nb', MultinomialNB(alpha=2.0)) #I experimented with diff value of alpha, found that alpha=2.0 is best accuracy
])


In [ ]:
model.fit(X_train, Y_train)

## **Make Prediction**

In [110]:
Y_pred = model.predict(X_test)

print(Y_pred[:10]) #print first 10 predictions to check its working

['ham' 'ham' 'ham' 'spam' 'ham' 'ham' 'spam' 'ham' 'spam' 'ham']


## **Evaluate Model**

In [111]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("Accuracy:", accuracy_score(Y_test, Y_pred))
print(confusion_matrix(Y_test, Y_pred))
print(classification_report(Y_test, Y_pred))
# Accuracy: 0.9856502242152466

Accuracy: 0.9856502242152466
[[965   1]
 [ 15 134]]
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.90      0.94       149

    accuracy                           0.99      1115
   macro avg       0.99      0.95      0.97      1115
weighted avg       0.99      0.99      0.99      1115



In [112]:
#Test with my own testcases to confirm its working

new_messages = [
    "Congratulations! You won a free iPhone",
    "Can we meet tomorrow at 3pm?",
    "Urgent! Your account will be suspended. Click now!"
]

predictions = model.predict(new_messages)

print(predictions)


['spam' 'ham' 'spam']
